# IFRNet - baseline evaluation + correction CNN (2 GPUs)

Split out of `vfi-batchprocessing`. **Needs the Kaggle accelerator "GPU T4 x2"** - the notebook asserts that two GPUs are visible and always uses both:

| Stage | How both GPUs are used |
|---|---|
| Frozen IFRNet inference (baseline eval + CNN training) | one independent IFRNet replica per GPU; every batch is split in half and the halves run concurrently |
| Correction CNN training | `nn.DataParallel` over `cuda:0` + `cuda:1` |

Run top to bottom. The quick check before the full training proves both GPUs are working.

In [2]:
import os

VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
IFRNET_ROOT = "/kaggle/input/datasets/jaisai002/ifrnet-kaggle"

clip = "00058/0745"
clip_dir = os.path.join(VIMEO_ROOT, "sequences", clip)

print("IFRNet files:")
print(os.listdir(IFRNET_ROOT))
print("\nClip:", clip_dir)
print(os.listdir(clip_dir))

for f in ["im1.png", "im2.png", "im3.png"]:
    p = os.path.join(clip_dir, f)
    print(f, os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else "")

IFRNet files:
['demo_8x.py', 'checkpoints', 'LICENSE', 'demo_2x.py', 'benchmarks', 'README.md', 'generate_flow.py', 'models', 'loss.py', 'utils.py', 'metric.py', 'train_gopro.py', 'figures', 'datasets.py', '.git', 'train_vimeo90k.py', 'liteflownet']

Clip: /kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet/sequences/00058/0745
['im3.png', 'im1.png', 'im2.png']
im1.png True 180710
im2.png True 181433
im3.png True 186152


In [3]:
import os

print(os.path.exists(
    "/kaggle/working/ifrnet_correction_generic_checkpoint.pth"
))

False


In [4]:
import os

base = "/kaggle/input/datasets"

for owner in os.listdir(base):
    print("\n", owner)
    path = os.path.join(base, owner)
    if os.path.isdir(path):
        for item in os.listdir(path):
            print("  ", item)


 hibro00
   vfi-csvs
   vfi-pths-backup
   vimeo-triplet-split

 jaisai002
   ifrnet-kaggle

 chenshu123
   vimeo-triplet


In [5]:
import os
import pandas as pd

SPLIT_ROOT = "/kaggle/input/datasets/hibro00/vimeo-triplet-split"

print("Split dataset contents:")
print(os.listdir(SPLIT_ROOT))

# Find the CSV automatically
csv_files = [
    f for f in os.listdir(SPLIT_ROOT)
    if f.lower().endswith(".csv")
]

print("\nCSV files:", csv_files)

assert len(csv_files) >= 1, "No CSV found in split dataset."

SPLITS_PATH = os.path.join(SPLIT_ROOT, csv_files[0])
splits = pd.read_csv(SPLITS_PATH)

print("\nUsing:", SPLITS_PATH)
print("Shape:", splits.shape)
print("\nColumns:", list(splits.columns))
print("\nSplit counts:")
print(splits["split"].value_counts())

print("\nFirst rows:")
display(splits.head())

Split dataset contents:
['splits.csv']

CSV files: ['splits.csv']

Using: /kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv
Shape: (55095, 2)

Columns: ['clip_id', 'split']

Split counts:
split
train          51324
test            3601
calibration      170
Name: count, dtype: int64

First rows:


,clip_id,split
0,00001/0001,train
1,00001/0002,train
2,00001/0003,train
3,00001/0004,train
4,00001/0005,train


In [6]:
# Validate the expected split structure

calib = splits[splits["split"] == "calibration"]
test = splits[splits["split"] == "test"]

print("Calibration:", len(calib))
print("Test:", len(test))

overlap = set(calib["clip_id"]) & set(test["clip_id"])

print("Calibration/Test overlap:", len(overlap))

assert len(calib) == 170, f"Expected 170 calibration clips, got {len(calib)}"
assert len(test) == 3601, f"Expected 3601 test clips, got {len(test)}"
assert len(overlap) == 0, "Calibration/test leakage detected!"

print("\nSPLIT VALIDATION PASSED")

Calibration: 170
Test: 3601
Calibration/Test overlap: 0

SPLIT VALIDATION PASSED


# ***IFRnet***

In [7]:
!pip install lpips -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.4 MB/s eta 0:00:00


In [8]:
# ============================================================
# 2-GPU SETUP - this notebook always runs on BOTH GPUs
# Kaggle: Session options -> Accelerator -> "GPU T4 x2"
# ============================================================
import contextlib
import os
import threading
import time

import numpy as np
import torch
import torch.nn as nn
from PIL import Image

NUM_GPUS = torch.cuda.device_count()
assert NUM_GPUS >= 2, (
    f"Only {NUM_GPUS} GPU(s) visible - this notebook needs two. "
    "Set the Kaggle accelerator to 'GPU T4 x2' and restart the session."
)

GPUS = [torch.device("cuda:0"), torch.device("cuda:1")]
GPU0, GPU1 = GPUS
DEVICE_IDS = [0, 1]                        # for nn.DataParallel
NUM_WORKERS = min(4, os.cpu_count() or 2)  # DataLoader workers
PER_GPU_BATCH = 4                          # global batch = 4 x 2 GPUs = 8

for d in GPUS:
    props = torch.cuda.get_device_properties(d)
    print(f"{d}: {props.name}, {props.total_memory / 1024**3:.1f} GB")


def gpu_ctx(dev):
    """Make `dev` the *current* CUDA device inside a `with` block.

    Needed because some code allocates on the index-less device torch.device("cuda"),
    which means "whichever GPU is current" (RIFE's Model.device() and its warp-grid cache,
    IFRNet's loss.py). Without this, everything silently ends up on cuda:0.
    """
    return torch.cuda.device(dev) if dev.type == "cuda" else contextlib.nullcontext()


def gpu_sync(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize(dev)


def gpu_snapshot():
    """Allocated bytes on each GPU (and reset the peak counters)."""
    snap = []
    for d in GPUS:
        if d.type == "cuda":
            torch.cuda.reset_peak_memory_stats(d)
            snap.append(torch.cuda.memory_allocated(d))
        else:
            snap.append(0)
    return snap


def gpu_report(before=None, min_extra_mb=None):
    """Print memory per GPU. With `before` (from gpu_snapshot) and `min_extra_mb`, assert that
    BOTH GPUs allocated real working memory since the snapshot, i.e. both were actually used."""
    for k, d in enumerate(GPUS):
        if d.type != "cuda":
            continue
        cur = torch.cuda.memory_allocated(d) / 2**20
        peak = torch.cuda.max_memory_allocated(d) / 2**20
        print(f"  {d}: {cur:8.1f} MiB allocated | {peak:8.1f} MiB peak")
        if before is not None and min_extra_mb is not None:
            extra = peak - before[k] / 2**20
            assert extra > min_extra_mb, (
                f"{d} only used {extra:.1f} MiB above its idle weights - it does not look like it did any work."
            )


def run_parallel(fns):
    """Run zero-argument callables concurrently (one thread each), return results in order.
    The first one runs in the calling thread. Same pattern as torch.nn.parallel.parallel_apply."""
    results, errors = [None] * len(fns), [None] * len(fns)

    def _wrap(i):
        try:
            results[i] = fns[i]()
        except BaseException as e:  # re-raised below, in the calling thread
            errors[i] = e

    threads = [threading.Thread(target=_wrap, args=(i,)) for i in range(1, len(fns))]
    for t in threads:
        t.start()
    _wrap(0)
    for t in threads:
        t.join()
    for e in errors:
        if e is not None:
            raise e
    return results


class MultiGPUInterpolator:
    """A frozen VFI model replicated once per GPU.

    run(im0, im1) splits the batch across the GPUs, runs the replicas concurrently and returns
    the middle frames concatenated on GPU0, clamped to [0, 1].
    run(..., timed=True) also returns the mean *per-clip* GPU time, measured on each replica's
    own GPU, so it stays comparable with the old single-GPU `inference_time_sec`.
    """

    def __init__(self, replicas, infer_fn, devices=GPUS):
        assert len(replicas) == len(devices)
        self.replicas, self.infer_fn, self.devices = replicas, infer_fn, devices

    def _one(self, k, x0, x1, timed):
        dev = self.devices[k]
        with gpu_ctx(dev), torch.no_grad():
            x0 = x0.to(dev, non_blocking=True)
            x1 = x1.to(dev, non_blocking=True)
            dt = 0.0
            if timed:
                gpu_sync(dev)
                t0 = time.perf_counter()
            out = self.infer_fn(self.replicas[k], x0, x1).clamp(0.0, 1.0)
            if timed:
                gpu_sync(dev)
                dt = time.perf_counter() - t0
        return out, dt, x0.shape[0]

    def run(self, im0, im1, timed=False):
        c0 = im0.chunk(len(self.devices))
        c1 = im1.chunk(len(self.devices))
        res = run_parallel([(lambda k=k: self._one(k, c0[k], c1[k], timed)) for k in range(len(c0))])
        out = torch.cat([r[0].to(self.devices[0]) for r in res], dim=0)
        if not timed:
            return out
        return out, sum(r[1] for r in res) / sum(r[2] for r in res)


def load_triplet(clip_id):
    """Vimeo triplet -> (im0, gt, im1), each (1, 3, H, W) float32 in [0, 1] on the CPU."""
    d = os.path.join(VIMEO_ROOT, "sequences", clip_id)
    frames = []
    for name in ("im1.png", "im2.png", "im3.png"):
        a = np.array(Image.open(os.path.join(d, name)).convert("RGB")).astype(np.float32) / 255.0
        frames.append(torch.from_numpy(a).permute(2, 0, 1).unsqueeze(0))
    return frames

cuda:0: Tesla T4, 14.6 GB
cuda:1: Tesla T4, 14.6 GB


In [9]:
# ============================================================
# LOAD ONE FROZEN IFRNet REPLICA PER GPU
# ============================================================
import sys

VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
IFRNET_ROOT = "/kaggle/input/datasets/jaisai002/ifrnet-kaggle"
IFRNET_CKPT = f"{IFRNET_ROOT}/checkpoints/IFRNet/IFRNet_Vimeo90K.pth"

if IFRNET_ROOT not in sys.path:
    sys.path.insert(0, IFRNET_ROOT)
from models.IFRNet import Model

print("PyTorch:", torch.__version__)


def load_ifrnet_replica(dev):
    with gpu_ctx(dev):
        m = Model()
        state = torch.load(IFRNET_CKPT, map_location=dev)
        if "state_dict" in state:
            state = state["state_dict"]
        m.load_state_dict(state, strict=True)
        m = m.to(dev).eval()
    for p in m.parameters():
        p.requires_grad_(False)  # frozen
    got = next(m.parameters()).device
    assert got == dev, f"IFRNet replica should be on {dev} but is on {got}"
    return m


def ifrnet_infer(model, x0, x1):
    # IFRNet needs an explicit time tensor (0.5 = middle frame), one value per sample
    embt = torch.full((x0.shape[0], 1, 1, 1), 0.5, dtype=x0.dtype, device=x0.device)
    return model.inference(x0, x1, embt)


ifrnet_replicas = [load_ifrnet_replica(d) for d in GPUS]
interp = MultiGPUInterpolator(ifrnet_replicas, ifrnet_infer)

for k in range(len(GPUS)):
    print(f"IFRNet replica {k}: parameters on", next(ifrnet_replicas[k].parameters()).device)
print("IFRNet loaded successfully on both GPUs.")
gpu_report()

PyTorch: 2.10.0+cu128
IFRNet replica 0: parameters on cuda:0
IFRNet replica 1: parameters on cuda:1
IFRNet loaded successfully on both GPUs.
  cuda:0:     19.1 MiB allocated |     38.1 MiB peak
  cuda:1:     19.1 MiB allocated |     38.1 MiB peak


In [10]:
import inspect
from models.IFRNet import Model

print(inspect.signature(Model.inference))

(self, img0, img1, embt, scale_factor=1.0)


In [11]:
# ============================================================
# IFRNet 2-GPU SMOKE TEST (one Vimeo triplet)
#   1) each replica runs the clip on its own GPU -> the two outputs must agree
#   2) interp.run() splits a batch across both GPUs -> same result again
#   3) PSNR / SSIM / LPIPS, as in the original smoke test
# ============================================================
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import lpips

test_ids = splits.loc[splits["split"] == "test", "clip_id"].tolist()
clip_id = test_ids[0]  # 00001/0389 in the original run

im0, gt, im1 = load_triplet(clip_id)
print("Clip:", clip_id, "| input shape:", tuple(im0.shape))

# --- 1) one replica per GPU ---------------------------------
preds, times = [], []
for k, dev in enumerate(GPUS):
    with gpu_ctx(dev), torch.no_grad():
        x0, x1 = im0.to(dev), im1.to(dev)
        interp.infer_fn(interp.replicas[k], x0, x1)  # warm-up
        gpu_sync(dev)
        t0 = time.perf_counter()
        pred = interp.infer_fn(interp.replicas[k], x0, x1).clamp(0, 1)
        gpu_sync(dev)
        times.append(time.perf_counter() - t0)
    assert pred.device == dev, f"output of replica {k} is on {pred.device}, expected {dev}"
    preds.append(pred.cpu())

gpu_diff = (preds[0] - preds[1]).abs().max().item()
assert gpu_diff < 1e-3, f"GPU0 and GPU1 disagree (max abs diff {gpu_diff}) - a replica is broken"

# --- 2) split-batch path ------------------------------------
out = interp.run(torch.cat([im0, im0]), torch.cat([im1, im1])).cpu()  # clip 0 -> GPU0, clip 1 -> GPU1
split_diff = max((out[i:i + 1] - preds[0]).abs().max().item() for i in range(2))
assert split_diff < 1e-3, f"split-batch output differs from single-clip output ({split_diff})"

# --- 3) metrics ---------------------------------------------
pred_np = preds[0][0].permute(1, 2, 0).numpy()
gt_np = gt[0].permute(1, 2, 0).numpy()
psnr = peak_signal_noise_ratio(gt_np, pred_np, data_range=1.0)
ssim = structural_similarity(gt_np, pred_np, data_range=1.0, channel_axis=-1)

lpips_model = lpips.LPIPS(net="alex").to(GPU0).eval()
with torch.no_grad():
    lp = lpips_model(preds[0].to(GPU0) * 2 - 1, gt.to(GPU0) * 2 - 1).item()

print("\n" + "=" * 50)
print("IFRNet 2-GPU SMOKE TEST")
print("=" * 50)
print("Clip :", clip_id)
print(f"Time : GPU0 {times[0]:.4f}s | GPU1 {times[1]:.4f}s")
print(f"GPU0 vs GPU1 max abs diff : {gpu_diff:.2e}")
print(f"split-batch max abs diff  : {split_diff:.2e}")
print(f"PSNR : {psnr:.4f} dB")
print(f"SSIM : {ssim:.6f}")
print(f"LPIPS: {lp:.6f}")
gpu_report()

Clip: 00001/0389 | input shape: (1, 3, 256, 448)
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 215MB/s] 


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth

IFRNet 2-GPU SMOKE TEST
Clip : 00001/0389
Time : GPU0 0.0209s | GPU1 0.0209s
GPU0 vs GPU1 max abs diff : 2.80e-06
split-batch max abs diff  : 2.95e-06
PSNR : 38.8865 dB
SSIM : 0.980218
LPIPS: 0.015842
  cuda:0:     28.6 MiB allocated |    104.9 MiB peak
  cuda:1:     23.7 MiB allocated |    110.0 MiB peak


#  **BATCH PROCESSING**

In [31]:
# # ============================================================
# # BASELINE BATCH EVALUATION on the test split - runs on BOTH GPUs
# # Each batch (4 clips per GPU = 8) is split in half; the halves run concurrently.
# # ============================================================
# import json
# import logging

# import pandas as pd
# from skimage.metrics import peak_signal_noise_ratio, structural_similarity
# from torch.utils.data import Dataset, DataLoader
# from tqdm.auto import tqdm
# import lpips as lpips_lib

# RUN_BASELINE_EVAL = True  # this cell was live in the original notebook; set False to skip the ~3601-clip evaluation
# MAX_EVAL_CLIPS = None             # e.g. 64 for a quick trial; None = all test clips

# CATALOG_DIR = "/kaggle/working/catalog"
# LOG_DIR = "/kaggle/working/logs"
# OUTPUT_PATH = f"{CATALOG_DIR}/ifrnet_metrics.csv"


# class VimeoTripletDataset(Dataset):
#     def __init__(self, clip_ids):
#         self.clip_ids = clip_ids

#     def __len__(self):
#         return len(self.clip_ids)

#     def __getitem__(self, idx):
#         clip_id = self.clip_ids[idx]
#         clip_dir = os.path.join(VIMEO_ROOT, "sequences", clip_id)

#         im0 = np.array(Image.open(f"{clip_dir}/im1.png").convert("RGB")).astype(np.float32) / 255.0
#         gt = np.array(Image.open(f"{clip_dir}/im2.png").convert("RGB")).astype(np.float32) / 255.0
#         im1 = np.array(Image.open(f"{clip_dir}/im3.png").convert("RGB")).astype(np.float32) / 255.0

#         t0 = torch.from_numpy(im0).permute(2, 0, 1)
#         t1 = torch.from_numpy(im1).permute(2, 0, 1)
#         tgt = torch.from_numpy(gt).permute(2, 0, 1)

#         return clip_id, t0, t1, tgt


# def evaluate_baseline(max_clips=None, per_gpu_batch=PER_GPU_BATCH, checkpoint_every=2):
#     os.makedirs(CATALOG_DIR, exist_ok=True)
#     os.makedirs(LOG_DIR, exist_ok=True)

#     # logging (file + console)
#     logger = logging.getLogger("ifrnet_batch")
#     logger.setLevel(logging.INFO)
#     logger.handlers.clear()
#     fh = logging.FileHandler(f"{LOG_DIR}/ifrnet_batch_run.log")
#     fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
#     ch = logging.StreamHandler()
#     ch.setFormatter(logging.Formatter("%(message)s"))
#     logger.addHandler(fh)
#     logger.addHandler(ch)

#     # test split (shared across all notebooks - never re-derived)
#     splits_df = pd.read_csv(SPLITS_PATH)
#     test_ids = splits_df.loc[splits_df["split"] == "test", "clip_id"].tolist()
#     if max_clips is not None:
#         test_ids = test_ids[:max_clips]
#     logger.info(f"Test clips to process: {len(test_ids)}")

#     lpips_model = lpips_lib.LPIPS(net="alex").to(GPU0).eval()
#     logger.info("LPIPS model loaded (GPU0).")

#     # resume support (failed clips get retried, not skipped)
#     if os.path.exists(OUTPUT_PATH):
#         done_df = pd.read_csv(OUTPUT_PATH)
#         done_ids = set(done_df.loc[done_df["status"] == "success", "clip_id"])
#         results = done_df[done_df["status"] == "success"].to_dict("records")
#         logger.info(f"Resuming: {len(done_ids)} successful clips already done.")
#     else:
#         done_ids = set()
#         results = []

#     remaining_ids = [c for c in test_ids if c not in done_ids]
#     logger.info(f"Remaining to process: {len(remaining_ids)}")

#     batch_size = per_gpu_batch * len(GPUS)
#     dl = DataLoader(
#         VimeoTripletDataset(remaining_ids),
#         batch_size=batch_size,
#         shuffle=False,
#         num_workers=NUM_WORKERS,
#         pin_memory=True,
#     )

#     pbar = tqdm(total=len(test_ids), initial=len(done_ids), unit="clip", desc="IFRNet eval (2 GPUs)")
#     processed = len(done_ids)
#     batch_start = time.time()

#     for batch_num, (clip_ids, im0_b, im1_b, gt_b) in enumerate(dl, start=1):
#         try:
#             # split across both GPUs; pred_b is on GPU0, clamped to [0, 1]
#             # per_clip_time = mean per-clip time on ONE gpu (comparable to the single-GPU numbers)
#             pred_b, per_clip_time = interp.run(im0_b, im1_b, timed=True)

#             with torch.no_grad():
#                 lpips_vals = (
#                     lpips_model(pred_b * 2.0 - 1.0, gt_b.to(GPU0, non_blocking=True) * 2.0 - 1.0)
#                     .flatten().detach().cpu().numpy()
#                 )

#             pred_np_batch = pred_b.detach().cpu().permute(0, 2, 3, 1).numpy()
#             gt_np_batch = gt_b.permute(0, 2, 3, 1).numpy()

#             for i, cid in enumerate(clip_ids):
#                 psnr = peak_signal_noise_ratio(gt_np_batch[i], pred_np_batch[i], data_range=1.0)
#                 ssim = structural_similarity(gt_np_batch[i], pred_np_batch[i], data_range=1.0, channel_axis=-1)
#                 results.append({
#                     "clip_id": cid,
#                     "psnr": float(psnr),
#                     "ssim": float(ssim),
#                     "lpips": float(lpips_vals[i]),
#                     "inference_time_sec": float(per_clip_time),
#                     "status": "success",
#                 })

#             processed += len(clip_ids)
#             pbar.update(len(clip_ids))

#             if batch_num % checkpoint_every == 0:
#                 pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False)
#                 elapsed = time.time() - batch_start
#                 rate = (processed - len(done_ids)) / elapsed if elapsed > 0 else 0
#                 eta = (len(test_ids) - processed) / rate if rate > 0 else float("nan")
#                 pbar.set_postfix_str(f"rate={rate:.2f} clip/s ETA={eta / 60:.1f}min")
#                 logger.info(
#                     f"[{processed}/{len(test_ids)}] elapsed={elapsed / 60:.1f}min "
#                     f"rate={rate:.2f} clips/s ETA={eta / 60:.1f}min"
#                 )

#         except Exception as e:
#             logger.error(f"BATCH FAILED (batch {batch_num}): {repr(e)}")
#             for cid in clip_ids:
#                 results.append({
#                     "clip_id": cid, "psnr": np.nan, "ssim": np.nan, "lpips": np.nan,
#                     "inference_time_sec": np.nan, "status": "failed",
#                 })
#             processed += len(clip_ids)
#             pbar.update(len(clip_ids))

#     pbar.close()

#     final_df = pd.DataFrame(results)
#     final_df.to_csv(OUTPUT_PATH, index=False)

#     logger.info("=" * 40)
#     logger.info("IFRNET BATCH COMPLETE")
#     logger.info("=" * 40)
#     logger.info(f"Successful: {(final_df['status'] == 'success').sum()}")
#     logger.info(f"Failed:     {(final_df['status'] == 'failed').sum()}")
#     logger.info(f"Mean PSNR:  {final_df['psnr'].mean():.4f} dB")
#     logger.info(f"Mean SSIM:  {final_df['ssim'].mean():.6f}")
#     logger.info(f"Mean LPIPS: {final_df['lpips'].mean():.6f}")
#     logger.info(f"Total time: {(time.time() - batch_start) / 60:.2f} minutes")
#     logger.info(f"Saved to:   {OUTPUT_PATH}")

#     # manifest (for joining with RAFT categories + the other VFI models later)
#     manifest = {
#         "model": "IFRNet",
#         "checkpoint": "IFRNet_Vimeo90K.pth",
#         "split_source": SPLITS_PATH,
#         "config": {"batch_size": batch_size, "per_gpu_batch": per_gpu_batch, "num_gpus": len(GPUS)},
#         "counts": {
#             "total": int(len(final_df)),
#             "success": int((final_df["status"] == "success").sum()),
#             "failed": int((final_df["status"] == "failed").sum()),
#         },
#         "files": {"metrics": OUTPUT_PATH, "log": f"{LOG_DIR}/ifrnet_batch_run.log"},
#     }
#     with open(f"{CATALOG_DIR}/ifrnet_manifest.json", "w") as f:
#         json.dump(manifest, f, indent=2)
#     logger.info(f"Saved manifest -> {CATALOG_DIR}/ifrnet_manifest.json")
#     return final_df


# if RUN_BASELINE_EVAL:
#     baseline_df = evaluate_baseline(max_clips=MAX_EVAL_CLIPS)
# else:
#     print("RUN_BASELINE_EVAL is False - skipping the baseline evaluation (call evaluate_baseline() to run it).")

# CORRECTION CNN  (trained on both GPUs)

In [12]:
# ============================================================
# CNN TRAINING DATA
# ============================================================

CLASSIFIED_PATH = "/kaggle/input/datasets/hibro00/vfi-csvs/raft_descriptors_train_classified.csv"

train_df = pd.read_csv(CLASSIFIED_PATH)

train_df = train_df[train_df["status"] == "success"].copy()

generic_df = train_df.copy()

rotation_df = train_df[
    train_df["predicted_category"] == "rotation"
].copy()

print("Generic training clips :", len(generic_df))
print("Rotation training clips:", len(rotation_df))

Generic training clips : 51324
Rotation training clips: 2980


In [13]:
# ============================================================
# LIGHTWEIGHT CORRECTION CNN
# Same residual-refinement architecture used for RIFE and FILM.
# Input: I0, I1, and frozen IFRNet output Ib (9 channels total).
# ============================================================

import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1),
        )

    def forward(self, x):
        return x + self.block(x)


class CorrectionCNN(nn.Module):
    def __init__(self, base_channels=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(9, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            ResidualBlock(base_channels),
            ResidualBlock(base_channels),
            ResidualBlock(base_channels),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.residual_head = nn.Conv2d(base_channels, 3, 3, padding=1)
        self.mask_head = nn.Sequential(
            nn.Conv2d(base_channels, 1, 3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, I0, I1, Ib):
        x = torch.cat([I0, I1, Ib], dim=1)
        features = self.encoder(x)
        residual = self.residual_head(features)
        mask = self.mask_head(features)
        corrected = torch.clamp(Ib + mask * residual, 0.0, 1.0)
        return corrected, residual, mask


In [14]:
# ============================================================
# CORRECTION DATASET
# ============================================================

from torch.utils.data import Dataset, DataLoader

class CorrectionDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        clip_id = self.df.iloc[idx]["clip_id"]

        clip_dir = os.path.join(
            VIMEO_ROOT,
            "sequences",
            clip_id
        )

        im0 = np.array(
            Image.open(
                os.path.join(clip_dir, "im1.png")
            ).convert("RGB")
        ).astype(np.float32) / 255.0

        gt = np.array(
            Image.open(
                os.path.join(clip_dir, "im2.png")
            ).convert("RGB")
        ).astype(np.float32) / 255.0

        im1 = np.array(
            Image.open(
                os.path.join(clip_dir, "im3.png")
            ).convert("RGB")
        ).astype(np.float32) / 255.0

        im0 = torch.from_numpy(im0).permute(2, 0, 1)
        gt  = torch.from_numpy(gt).permute(2, 0, 1)
        im1 = torch.from_numpy(im1).permute(2, 0, 1)

        return clip_id, im0, im1, gt

In [15]:
# ============================================================
# LOSS
# ============================================================

def charbonnier_loss(pred, target, eps=1e-3):
    return torch.mean(
        torch.sqrt((pred - target) ** 2 + eps ** 2)
    )


def correction_loss(corrected, gt, residual):
    l1 = torch.mean(torch.abs(corrected - gt))

    charbonnier = charbonnier_loss(corrected, gt)

    residual_reg = torch.mean(torch.abs(residual))

    loss = (
        l1
        + 0.1 * charbonnier
        + 0.001 * residual_reg
    )

    return loss

In [16]:
# ============================================================
# TRAIN CORRECTION CNN - BOTH T4 GPUs
# ============================================================

from tqdm import tqdm


def batch_psnr_sum(a, b):
    mse = (
        ((a - b) ** 2)
        .flatten(1)
        .mean(1)
        .clamp_min(1e-10)
    )

    return (
        10.0 * torch.log10(1.0 / mse)
    ).sum()


def train_correction(
    dataframe,
    save_name,
    interp,
    epochs=5,
    per_gpu_batch=PER_GPU_BATCH,
    lr=1e-4,
    num_workers=NUM_WORKERS,
):

    dataset = CorrectionDataset(dataframe)

    global_batch = (
        per_gpu_batch * len(GPUS)
    )

    loader = DataLoader(
        dataset,
        batch_size=global_batch,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )

    assert len(loader) > 0, (
        f"{len(dataset)} clips is fewer than "
        f"one global batch ({global_batch})"
    )

    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    correction_model = (
        CorrectionCNN()
        .to(GPU0)
    )

    parallel_model = nn.DataParallel(
        correction_model,
        device_ids=DEVICE_IDS,
        output_device=0
    )

    optimizer = torch.optim.Adam(
        correction_model.parameters(),
        lr=lr
    )

    # --------------------------------------------------------
    # FILES
    # --------------------------------------------------------

    final_path = (
        f"/kaggle/working/"
        f"{save_name}.pth"
    )

    checkpoint_path = (
        f"/kaggle/working/"
        f"{save_name}_checkpoint.pth"
    )

    history_path = (
        f"/kaggle/working/"
        f"{save_name}_history.pth"
    )

    # --------------------------------------------------------
    # RESUME
    # --------------------------------------------------------

    start_epoch = 0
    history = []

    if os.path.exists(checkpoint_path):

        print("\nCheckpoint found:")
        print(checkpoint_path)

        checkpoint = torch.load(
            checkpoint_path,
            map_location=GPU0
        )

        correction_model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = (
            checkpoint["epoch"] + 1
        )

        history = checkpoint.get(
            "history",
            []
        )

        print(
            f"Resuming from epoch "
            f"{start_epoch + 1}/{epochs}"
        )

    else:

        print("\nNo checkpoint found.")
        print("Starting from epoch 1.")

    # --------------------------------------------------------
    # INFO
    # --------------------------------------------------------

    print("\nTraining:", save_name)

    print(
        f"Clips: {len(dataset)}"
    )

    print(
        f"Global batch: {global_batch} "
        f"({per_gpu_batch} per GPU x "
        f"{len(GPUS)} GPUs)"
    )

    print(
        f"Steps/epoch: {len(loader)}"
    )

    print(
        f"Epochs remaining: "
        f"{epochs - start_epoch}"
    )

    print(
        "GPUs:",
        [str(d) for d in GPUS]
    )

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------

    for epoch in range(
        start_epoch,
        epochs
    ):

        parallel_model.train()

        epoch_loss = 0.0
        seen = 0

        psnr_base = torch.zeros(
            (),
            device=GPU0
        )

        psnr_corr = torch.zeros(
            (),
            device=GPU0
        )

        pbar = tqdm(
            loader,
            desc=(
                f"Epoch "
                f"{epoch + 1}/{epochs}"
            ),
            unit="batch",
            dynamic_ncols=True
        )

        for (
            clip_ids,
            im0,
            im1,
            gt
        ) in pbar:

            try:

                # ------------------------------------------------
                # IFRNet - BOTH GPUs
                # ------------------------------------------------

                with torch.no_grad():

                    ib = interp.run(
                        im0,
                        im1
                    )

                # ------------------------------------------------
                # MOVE DATA TO GPU 0
                # ------------------------------------------------

                im0 = im0.to(
                    GPU0,
                    non_blocking=True
                )

                im1 = im1.to(
                    GPU0,
                    non_blocking=True
                )

                gt = gt.to(
                    GPU0,
                    non_blocking=True
                )

                # ------------------------------------------------
                # CORRECTION CNN - BOTH GPUs
                # ------------------------------------------------

                optimizer.zero_grad(
                    set_to_none=True
                )

                corrected, residual, mask = parallel_model(
                    im0,
                    im1,
                    ib,
                )

                # ------------------------------------------------
                # LOSS
                # ------------------------------------------------

                loss = correction_loss(
                    corrected,
                    gt,
                    residual
                )

                if not torch.isfinite(loss):

                    print(
                        "\nNon-finite loss."
                        " Skipping batch."
                    )

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    continue

                # ------------------------------------------------
                # BACKPROP
                # ------------------------------------------------

                loss.backward()

                optimizer.step()

                # ------------------------------------------------
                # STATS
                # ------------------------------------------------

                loss_value = loss.item()

                epoch_loss += loss_value

                with torch.no_grad():

                    psnr_base += (
                        batch_psnr_sum(
                            ib,
                            gt
                        )
                    )

                    psnr_corr += (
                        batch_psnr_sum(
                            corrected.detach(),
                            gt
                        )
                    )

                seen += gt.shape[0]

                running_loss = (
                    epoch_loss /
                    (pbar.n + 1)
                )

                pbar.set_postfix(
                    loss=f"{loss_value:.5f}",
                    avg=f"{running_loss:.5f}"
                )

            except RuntimeError as e:

                if "out of memory" in str(e).lower():

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    torch.cuda.empty_cache()

                    raise RuntimeError(
                        "\nCUDA OOM.\n"
                        f"Global batch: "
                        f"{global_batch}\n"
                        f"Per-GPU batch: "
                        f"{per_gpu_batch}\n"
                        "Reduce per_gpu_batch."
                    )

                raise

        # --------------------------------------------------------
        # EPOCH RESULTS
        # --------------------------------------------------------

        rec = {
            "epoch": epoch + 1,

            "loss": (
                epoch_loss /
                len(loader)
            ),

            "psnr_base": (
                psnr_base /
                seen
            ).item(),

            "psnr_corrected": (
                psnr_corr /
                seen
            ).item(),
        }

        history.append(rec)

        improvement = (
            rec["psnr_corrected"]
            - rec["psnr_base"]
        )

        print()
        print("=" * 70)

        print(
            f"Epoch {rec['epoch']}/{epochs} COMPLETE"
        )

        print(
            f"Loss: {rec['loss']:.6f}"
        )

        print(
            f"PSNR: "
            f"{rec['psnr_base']:.3f} dB"
            f" -> "
            f"{rec['psnr_corrected']:.3f} dB"
            f" ({improvement:+.3f} dB)"
        )

        # --------------------------------------------------------
        # SAVE CHECKPOINT
        # --------------------------------------------------------

        checkpoint = {
            "epoch": epoch,

            "model_state_dict":
                correction_model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "history":
                history,

            "save_name":
                save_name,

            "per_gpu_batch":
                per_gpu_batch,

            "global_batch":
                global_batch,
        }

        torch.save(
            checkpoint,
            checkpoint_path
        )

        # Save history separately too
        torch.save(
            history,
            history_path
        )

        # Save current model
        torch.save(
            correction_model.state_dict(),
            final_path
        )

        print(
            "\nCHECKPOINT SAVED:"
        )

        print(
            checkpoint_path
        )

        print(
            "\nMODEL SAVED:"
        )

        print(
            final_path
        )

        print("=" * 70)

        # --------------------------------------------------------
        # MEMORY
        # --------------------------------------------------------

        torch.cuda.empty_cache()

    print()
    print("=" * 70)
    print("TRAINING COMPLETE")
    print("=" * 70)

    print(
        "Final model:",
        final_path
    )

    print(
        "Checkpoint:",
        checkpoint_path
    )

    return correction_model, history

In [20]:
# ============================================================
# QUICK CHECK (~1 min): 1 epoch on 64 clips, then prove BOTH GPUs did work
# ============================================================
before = gpu_snapshot()

smoke_df = generic_df.sample(n=min(64, len(generic_df)), random_state=0)
_ = train_correction(smoke_df, "_smoke_test", interp, epochs=1)

print("\nGPU memory after the quick check:")
gpu_report(before, min_extra_mb=50)  # asserts that BOTH GPUs used > 50 MiB above their idle weights
print("Both GPUs are in use.")


No checkpoint found.
Starting from epoch 1.

Training: _smoke_test
Clips: 64
Global batch: 8 (4 per GPU x 2 GPUs)
Steps/epoch: 8
Epochs remaining: 1
GPUs: ['cuda:0', 'cuda:1']


Epoch 1/1: 100%|██████████| 8/8 [00:03<00:00,  2.64batch/s, avg=0.01915, loss=0.01631]



Epoch 1/1 COMPLETE
Loss: 0.019154
PSNR: 35.691 dB -> 31.953 dB (-3.738 dB)

CHECKPOINT SAVED:
/kaggle/working/_smoke_test_checkpoint.pth

MODEL SAVED:
/kaggle/working/_smoke_test.pth

TRAINING COMPLETE
Final model: /kaggle/working/_smoke_test.pth
Checkpoint: /kaggle/working/_smoke_test_checkpoint.pth

GPU memory after the quick check:
  cuda:0:     28.8 MiB allocated |    659.1 MiB peak
  cuda:1:     23.7 MiB allocated |    572.0 MiB peak
Both GPUs are in use.


In [17]:
import shutil
import os

src = "/kaggle/input/datasets/hibro00/vfi-pths-backup/ifrnet_correction_generic_checkpoint (3).pth"
dst = "/kaggle/working/ifrnet_correction_generic_checkpoint.pth"

shutil.copy2(src, dst)

print("Copied:", os.path.exists(dst))
print(dst)

Copied: True
/kaggle/working/ifrnet_correction_generic_checkpoint.pth


In [40]:
# # ============================================================
# # FULL TRAINING - generic correction CNN (all training clips)
# # ============================================================
# EPOCHS = 5
# LR = 1e-4

# generic_cnn, generic_history = train_correction(
#     generic_df,
#     "ifrnet_correction_generic",
#     interp,
#     epochs=EPOCHS,
#     per_gpu_batch=PER_GPU_BATCH,
#     lr=LR,
# )


Checkpoint found:
/kaggle/working/ifrnet_correction_generic_checkpoint.pth
Resuming from epoch 5/5

Training: ifrnet_correction_generic
Clips: 51324
Global batch: 8 (4 per GPU x 2 GPUs)
Steps/epoch: 6415
Epochs remaining: 1
GPUs: ['cuda:0', 'cuda:1']


Epoch 5/5: 100%|██████████| 6415/6415 [15:41<00:00,  6.82batch/s, avg=0.00835, loss=0.00799]



Epoch 5/5 COMPLETE
Loss: 0.008353
PSNR: 37.010 dB -> 37.048 dB (+0.038 dB)

CHECKPOINT SAVED:
/kaggle/working/ifrnet_correction_generic_checkpoint.pth

MODEL SAVED:
/kaggle/working/ifrnet_correction_generic.pth

TRAINING COMPLETE
Final model: /kaggle/working/ifrnet_correction_generic.pth
Checkpoint: /kaggle/working/ifrnet_correction_generic_checkpoint.pth


In [18]:
print(len(rotation_df))

2980


In [22]:
EPOCHS = 5
PER_GPU_BATCH = 4
LR = 1e-4

In [23]:
# ============================================================
# FULL TRAINING - rotation-specialised correction CNN
# ============================================================
rotation_cnn, rotation_history = train_correction(
    rotation_df,
    "ifrnet_correction_rotation",
    interp,
    epochs=EPOCHS,
    per_gpu_batch=PER_GPU_BATCH,
    lr=LR,
)


No checkpoint found.
Starting from epoch 1.

Training: ifrnet_correction_rotation
Clips: 2980
Global batch: 8 (4 per GPU x 2 GPUs)
Steps/epoch: 372
Epochs remaining: 5
GPUs: ['cuda:0', 'cuda:1']


Epoch 1/5: 100%|██████████| 372/372 [00:52<00:00,  7.03batch/s, avg=0.00949, loss=0.00890]



Epoch 1/5 COMPLETE
Loss: 0.009489
PSNR: 34.876 dB -> 34.857 dB (-0.019 dB)

CHECKPOINT SAVED:
/kaggle/working/ifrnet_correction_rotation_checkpoint.pth

MODEL SAVED:
/kaggle/working/ifrnet_correction_rotation.pth


Epoch 2/5: 100%|██████████| 372/372 [00:53<00:00,  6.98batch/s, avg=0.00941, loss=0.01197]



Epoch 2/5 COMPLETE
Loss: 0.009409
PSNR: 34.884 dB -> 34.883 dB (-0.001 dB)

CHECKPOINT SAVED:
/kaggle/working/ifrnet_correction_rotation_checkpoint.pth

MODEL SAVED:
/kaggle/working/ifrnet_correction_rotation.pth


...


Epoch 3/5 COMPLETE
Loss: 0.009413
PSNR: 34.872 dB -> 34.872 dB (-0.000 dB)

CHECKPOINT SAVED:
/kaggle/working/ifrnet_correction_rotation_checkpoint.pth

MODEL SAVED:
/kaggle/working/ifrnet_correction_rotation.pth


Epoch 4/5: 100%|██████████| 372/372 [00:53<00:00,  6.99batch/s, avg=0.00941, loss=0.01160]



Epoch 4/5 COMPLETE
Loss: 0.009406
PSNR: 34.882 dB -> 34.883 dB (+0.000 dB)

CHECKPOINT SAVED:
/kaggle/working/ifrnet_correction_rotation_checkpoint.pth

MODEL SAVED:
/kaggle/working/ifrnet_correction_rotation.pth


Epoch 5/5: 100%|██████████| 372/372 [00:52<00:00,  7.03batch/s, avg=0.00941, loss=0.01204]


Epoch 5/5 COMPLETE
Loss: 0.009406
PSNR: 34.877 dB -> 34.877 dB (+0.001 dB)

CHECKPOINT SAVED:
/kaggle/working/ifrnet_correction_rotation_checkpoint.pth

MODEL SAVED:
/kaggle/working/ifrnet_correction_rotation.pth

TRAINING COMPLETE
Final model: /kaggle/working/ifrnet_correction_rotation.pth
Checkpoint: /kaggle/working/ifrnet_correction_rotation_checkpoint.pth


In [27]:
# ============================================================
# IFRNet ROTATION MODEL - RELOAD CHECK
# ============================================================

def load_correction(path):
    net = CorrectionCNN().to(GPU0)
    net.load_state_dict(
        torch.load(path, map_location=GPU0)
    )
    return net.eval()


def psnr_t(a, b):
    mse = torch.mean((a - b) ** 2).clamp_min(1e-10)
    return (10.0 * torch.log10(1.0 / mse)).item()


test_ids = splits.loc[
    splits["split"] == "test",
    "clip_id"
].tolist()

im0, gt, im1 = load_triplet(test_ids[0])

im0 = im0.to(GPU0)
im1 = im1.to(GPU0)
gt = gt.to(GPU0)

with torch.no_grad():

    # Baseline IFRNet
    ib = interp.run(im0, im1)

    base_psnr = psnr_t(ib, gt)

    print(
        f"clip {test_ids[0]} | "
        f"base IFRNet PSNR: {base_psnr:.3f} dB"
    )

    # ============================================================
    # IFRNet ROTATION CORRECTION TEST
    # ============================================================
    
    net = load_correction(
        "/kaggle/working/ifrnet_correction_rotation.pth"
    )
    
    with torch.no_grad():
    
        corrected, _, _ = net(im0, im1, ib)
    
        corrected_psnr = psnr_t(
            corrected,
            gt
        )
    
    print(
        f"  + rotation CNN PSNR: "
        f"{corrected_psnr:.3f} dB"
    )
    
    print(
        f"  Improvement: "
        f"{corrected_psnr - base_psnr:+.3f} dB"
    )

clip 00001/0389 | base IFRNet PSNR: 38.887 dB
  + rotation CNN PSNR: 38.869 dB
  Improvement: -0.018 dB


In [40]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from skimage.metrics import structural_similarity as ssim
import lpips

# ============================================================
# PATHS
# ============================================================

VIMEO_ROOT = "/kaggle/input/datasets/chenshu123/vimeo-triplet/vimeo_triplet"
SPLITS_PATH = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"

BASELINE_CSV = "/kaggle/input/datasets/hibro00/vfi-csvs/ifrnet_metrics.csv"
MODEL_PATH = "/kaggle/working/ifrnet_correction_rotation.pth"

OUT_CSV = "/kaggle/working/ifrnet_targeted_rotation_results.csv"

TEST_CLASSIFIED = "/kaggle/input/datasets/hibro00/vfi-csvs/test_motion_classifications.csv"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Model:", MODEL_PATH)
print("Baseline:", BASELINE_CSV)


# ============================================================
# LOAD BASELINE
# ============================================================

baseline_df = pd.read_csv(BASELINE_CSV)

baseline_df["clip_id"] = baseline_df["clip_id"].astype(str)

print("\nOriginal baseline rows:", len(baseline_df))
print("Baseline columns:", baseline_df.columns.tolist())


# ============================================================
# LOAD FINAL SPLITS
# ============================================================

splits = pd.read_csv(SPLITS_PATH)

print("\nSplit columns:", splits.columns.tolist())

splits["clip_id"] = splits["clip_id"].astype(str)

test_ids = (
    splits.loc[
        splits["split"].astype(str).str.lower() == "test",
        "clip_id"
    ]
    .tolist()
)

test_ids = list(dict.fromkeys(test_ids))

print("Final test clips:", len(test_ids))


# ============================================================
# FILTER OLD BASELINE TO FINAL TEST SET
# ============================================================

baseline_df = baseline_df[
    baseline_df["clip_id"].isin(test_ids)
].copy()

print(
    "Baseline rows after final-test filtering:",
    len(baseline_df)
)


# ============================================================
# LOAD CLASSIFIED TEST DATA
# ============================================================

test_classified = pd.read_csv(TEST_CLASSIFIED)

test_classified["clip_id"] = (
    test_classified["clip_id"].astype(str)
)

test_classified = test_classified[
    test_classified["clip_id"].isin(test_ids)
].copy()

print("\nClassified test rows:", len(test_classified))

print("\nCategory counts:")
print(
    test_classified["predicted_category"].value_counts()
)


# ============================================================
# LOAD ROTATION CORRECTION MODEL
# ============================================================

net = CorrectionCNN(base_channels=32)

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):
    state_dict = checkpoint["model_state_dict"]
else:
    state_dict = checkpoint

# Remove DataParallel prefix if present
state_dict = {
    k.replace("module.", "", 1)
    if k.startswith("module.")
    else k: v
    for k, v in state_dict.items()
}

net.load_state_dict(
    state_dict,
    strict=True
)

net = net.to(DEVICE)
net.eval()

print("\nRotation correction model loaded.")


# ============================================================
# LPIPS
# ============================================================

lpips_model = lpips.LPIPS(
    net="alex"
).to(DEVICE)

lpips_model.eval()


# ============================================================
# LOAD VIMEO TRIPLET
# ============================================================

def load_triplet(clip_id):

    clip_id = str(clip_id)

    parts = clip_id.split("/")

    if len(parts) != 2:
        raise ValueError(
            f"Unexpected clip ID format: {clip_id}"
        )

    clip_dir = os.path.join(
        VIMEO_ROOT,
        "sequences",
        parts[0],
        parts[1]
    )

    im0 = Image.open(
        os.path.join(clip_dir, "im1.png")
    ).convert("RGB")

    gt = Image.open(
        os.path.join(clip_dir, "im2.png")
    ).convert("RGB")

    im1 = Image.open(
        os.path.join(clip_dir, "im3.png")
    ).convert("RGB")

    def to_tensor(img):

        arr = np.asarray(
            img,
            dtype=np.float32
        ) / 255.0

        return torch.from_numpy(
            arr
        ).permute(
            2, 0, 1
        ).unsqueeze(0)

    return (
        to_tensor(im0),
        to_tensor(gt),
        to_tensor(im1)
    )


# ============================================================
# METRICS
# ============================================================

def calc_psnr(pred, target):

    mse = torch.mean(
        (pred - target) ** 2
    )

    if mse.item() == 0:
        return 100.0

    return float(
        (-10.0 * torch.log10(mse)).item()
    )


def calc_ssim(pred, target):

    pred_np = (
        pred.squeeze(0)
        .detach()
        .cpu()
        .numpy()
        .transpose(1, 2, 0)
    )

    target_np = (
        target.squeeze(0)
        .detach()
        .cpu()
        .numpy()
        .transpose(1, 2, 0)
    )

    return float(
        ssim(
            target_np,
            pred_np,
            channel_axis=2,
            data_range=1.0
        )
    )


# ============================================================
# IFRNET
# ============================================================

print(
    "\nUsing IFRNet interpolator:",
    type(interp)
)


# ============================================================
# EVALUATION
# ============================================================

results = []

print("\n========================================")
print("IFRNET TARGETED ROTATION EVALUATION")
print("========================================")
print("Total test clips:", len(test_ids))
print("========================================\n")


with torch.no_grad():

    for i, clip_id in enumerate(test_ids):

        # ----------------------------------------------------
        # LOAD
        # ----------------------------------------------------

        im0, gt, im1 = load_triplet(
            clip_id
        )

        im0 = im0.to(DEVICE)
        gt = gt.to(DEVICE)
        im1 = im1.to(DEVICE)

        # ----------------------------------------------------
        # IFRNET BASELINE INTERPOLATION
        # ----------------------------------------------------

        ib = interp.run(
            im0,
            im1
        )

        if not torch.is_tensor(ib):
            ib = torch.from_numpy(ib)

        ib = ib.to(DEVICE)

        if ib.dim() == 3:
            ib = ib.unsqueeze(0)

        ib = torch.clamp(
            ib,
            0,
            1
        )

        # ----------------------------------------------------
        # ROTATION CORRECTION
        #
        # IFRNet CorrectionCNN receives I0, I1, and Ib directly.
        # It concatenates them internally as a 9-channel tensor.
        # ----------------------------------------------------

        corrected, residual, mask = net(im0, im1, ib)

        corrected = torch.clamp(
            corrected,
            0,
            1
        )

        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        corrected_psnr = calc_psnr(
            corrected,
            gt
        )

        corrected_ssim = calc_ssim(
            corrected,
            gt
        )

        corrected_lpips = float(
            lpips_model(
                corrected * 2 - 1,
                gt * 2 - 1
            ).mean().item()
        )

        results.append({
            "clip_id": clip_id,
            "corrected_psnr": corrected_psnr,
            "corrected_ssim": corrected_ssim,
            "corrected_lpips": corrected_lpips
        })

        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if (
            (i + 1) % 100 == 0
            or i == 0
        ):

            print(
                f"{i+1}/{len(test_ids)} | "
                f"{clip_id} | "
                f"PSNR: {corrected_psnr:.3f} | "
                f"SSIM: {corrected_ssim:.5f} | "
                f"LPIPS: {corrected_lpips:.5f}"
            )


# ============================================================
# CORRECTED DATAFRAME
# ============================================================

corrected_df = pd.DataFrame(
    results
)

corrected_df["clip_id"] = (
    corrected_df["clip_id"].astype(str)
)

print(
    "\nCorrected rows:",
    len(corrected_df)
)


# ============================================================
# MERGE BASELINE + CORRECTED
# ============================================================

merged = baseline_df.merge(
    corrected_df,
    on="clip_id",
    how="inner"
)


# ============================================================
# ADD CATEGORY
# ============================================================

category_df = (
    test_classified[
        ["clip_id", "predicted_category"]
    ]
    .drop_duplicates("clip_id")
)

merged = merged.merge(
    category_df,
    on="clip_id",
    how="left"
)

merged = merged.rename(
    columns={"predicted_category": "category"}
)
# ============================================================
# DELTAS
# ============================================================

merged["psnr_delta"] = (
    merged["corrected_psnr"]
    - merged["psnr"]
)

merged["ssim_delta"] = (
    merged["corrected_ssim"]
    - merged["ssim"]
)

merged["lpips_delta"] = (
    merged["corrected_lpips"]
    - merged["lpips"]
)


# ============================================================
# SAVE
# ============================================================

merged.to_csv(
    OUT_CSV,
    index=False
)

print("\n========================================")
print("FINAL IFRNET TARGETED EVALUATION")
print("========================================")
print("Merged rows:", len(merged))
print("Saved:", OUT_CSV)


# ============================================================
# CATEGORY SUMMARY
# ============================================================

summary = (
    merged
    .groupby("category")
    .agg(

        n=("clip_id", "count"),

        baseline_psnr=("psnr", "mean"),
        corrected_psnr=("corrected_psnr", "mean"),
        psnr_delta=("psnr_delta", "mean"),

        baseline_ssim=("ssim", "mean"),
        corrected_ssim=("corrected_ssim", "mean"),
        ssim_delta=("ssim_delta", "mean"),

        baseline_lpips=("lpips", "mean"),
        corrected_lpips=("corrected_lpips", "mean"),
        lpips_delta=("lpips_delta", "mean")
    )
    .reset_index()
)


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n========================================")
print("CATEGORY RESULTS")
print("========================================")

display(summary)


print("\n========================================")
print("ROTATION RESULT")
print("========================================")

display(
    summary[
        summary["category"]
        .astype(str)
        .str.lower()
        == "rotation"
    ]
)


print("\n========================================")
print("NON-TARGET RESULTS")
print("========================================")

display(
    summary[
        summary["category"]
        .astype(str)
        .str.lower()
        != "rotation"
    ]
)

Device: cuda
Model: /kaggle/working/ifrnet_correction_rotation.pth
Baseline: /kaggle/input/datasets/hibro00/vfi-csvs/ifrnet_metrics.csv

Original baseline rows: 3732
Baseline columns: ['clip_id', 'psnr', 'ssim', 'lpips', 'inference_time_sec', 'status']

Split columns: ['clip_id', 'split']
Final test clips: 3601
Baseline rows after final-test filtering: 3601

Classified test rows: 3601

Category counts:
predicted_category
near_linear    1928
non_rigid      1171
occlusion       322
rotation        180
Name: count, dtype: int64

Rotation correction model loaded.
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth

Using IFRNet interpolator: <class '__main__.MultiGPUInterpolator'>

IFRNET TARGETED ROTATION EVALUATION
Total test clips: 3601

1/3601 | 00001/0389 | PSNR: 38.869 | SSIM: 0.98015 | LPIPS: 0.01578
100/3601 | 00003/0436 | PSNR: 36.013 | SSIM: 0.97313 | LPIPS: 0.02508
200/3601 | 00005/0258 | PSNR: 37.597 | SSIM: 0.98668 | LPIPS: 0.00652
300/3601 | 00007/0017 | PSNR: 34.460 | SSIM: 0.94919 | LPIPS: 0.03170
400/3601 | 00008/0179 | PSNR: 38.850 | SSIM: 0.96372 | LPIPS: 0.02711
500/3601 | 00010/0207 | PSNR: 40.201 | SSIM: 0.98490 | LPIPS: 0.01090
600/3601 | 00012/0457 | PSNR: 37.161 | SSIM: 0.97253 | LPIPS: 0.01122
700/3601 | 00013/0988 | PSNR: 29.110 | SSIM: 0.94243 | LPIPS: 0.01411
800/3601 | 00016/0554 | PSNR: 36.996 | SSIM: 0.96613 | LPIPS: 0.04947
900/3601 | 00018/0731 | PSNR: 29.547 | SSIM: 0.96686 | LPIPS: 0.02606
1000/3601 | 00020/0199 | PSNR: 36.601 | SSIM: 0.98372 | LPIPS: 0.00844
1100/3601 | 00

KeyError: "['cpredicted_ategory'] not in index"

In [41]:
# ============================================================
# MERGE BASELINE + CORRECTED
# ============================================================

merged = baseline_df.merge(
    corrected_df,
    on="clip_id",
    how="inner"
)

print("Merged rows:", len(merged))


# ============================================================
# ADD CATEGORY
# ============================================================

category_df = (
    test_classified[
        ["clip_id", "predicted_category"]
    ]
    .drop_duplicates("clip_id")
)

merged = merged.merge(
    category_df,
    on="clip_id",
    how="left"
)

merged = merged.rename(
    columns={
        "predicted_category": "category"
    }
)


# ============================================================
# DELTAS
# ============================================================

merged["psnr_delta"] = (
    merged["corrected_psnr"]
    - merged["psnr"]
)

merged["ssim_delta"] = (
    merged["corrected_ssim"]
    - merged["ssim"]
)

merged["lpips_delta"] = (
    merged["corrected_lpips"]
    - merged["lpips"]
)


# ============================================================
# SAVE
# ============================================================

OUT_CSV = "/kaggle/working/ifrnet_targeted_rotation_results.csv"

merged.to_csv(
    OUT_CSV,
    index=False
)

print("\n========================================")
print("FINAL IFRNET TARGETED EVALUATION")
print("========================================")
print("Rows:", len(merged))
print("Saved:", OUT_CSV)


# ============================================================
# CATEGORY SUMMARY
# ============================================================

summary = (
    merged
    .groupby("category")
    .agg(

        n=("clip_id", "count"),

        baseline_psnr=("psnr", "mean"),
        corrected_psnr=("corrected_psnr", "mean"),
        psnr_delta=("psnr_delta", "mean"),

        baseline_ssim=("ssim", "mean"),
        corrected_ssim=("corrected_ssim", "mean"),
        ssim_delta=("ssim_delta", "mean"),

        baseline_lpips=("lpips", "mean"),
        corrected_lpips=("corrected_lpips", "mean"),
        lpips_delta=("lpips_delta", "mean")
    )
    .reset_index()
)


# ============================================================
# RESULTS
# ============================================================

print("\n========================================")
print("CATEGORY RESULTS")
print("========================================")

display(summary)


# ============================================================
# ROTATION
# ============================================================

print("\n========================================")
print("ROTATION RESULT")
print("========================================")

rotation_result = summary[
    summary["category"]
    .astype(str)
    .str.lower()
    == "rotation"
]

display(rotation_result)


# ============================================================
# NON-TARGET
# ============================================================

print("\n========================================")
print("NON-TARGET RESULTS")
print("========================================")

non_target = summary[
    summary["category"]
    .astype(str)
    .str.lower()
    != "rotation"
]

display(non_target)

Merged rows: 3601

FINAL IFRNET TARGETED EVALUATION
Rows: 3601
Saved: /kaggle/working/ifrnet_targeted_rotation_results.csv

CATEGORY RESULTS


,category,n,baseline_psnr,corrected_psnr,psnr_delta,baseline_ssim,corrected_ssim,ssim_delta,baseline_lpips,corrected_lpips,lpips_delta
0,near_linear,1928,36.749227,36.748863,-0.000364,0.971249,0.971264,0.000014,0.017902,0.017842,-0.000060
1,non_rigid,1171,34.900728,34.900241,-0.000487,0.966322,0.966329,0.000006,0.021730,0.021668,-0.000062
2,occlusion,322,35.007154,35.007552,0.000398,0.965142,0.965163,0.000020,0.022874,0.022807,-0.000067
3,rotation,180,32.989862,32.990301,0.000438,0.959542,0.959566,0.000024,0.026671,0.026588,-0.000082



ROTATION RESULT


,category,n,baseline_psnr,corrected_psnr,psnr_delta,baseline_ssim,corrected_ssim,ssim_delta,baseline_lpips,corrected_lpips,lpips_delta
3,rotation,180,32.989862,32.990301,0.000438,0.959542,0.959566,0.000024,0.026671,0.026588,-0.000082



NON-TARGET RESULTS


,category,n,baseline_psnr,corrected_psnr,psnr_delta,baseline_ssim,corrected_ssim,ssim_delta,baseline_lpips,corrected_lpips,lpips_delta
0,near_linear,1928,36.749227,36.748863,-0.000364,0.971249,0.971264,0.000014,0.017902,0.017842,-0.000060
1,non_rigid,1171,34.900728,34.900241,-0.000487,0.966322,0.966329,0.000006,0.021730,0.021668,-0.000062
2,occlusion,322,35.007154,35.007552,0.000398,0.965142,0.965163,0.000020,0.022874,0.022807,-0.000067


In [42]:
# ============================================================
# LOAD GENERIC CORRECTION MODEL (from the backed-up checkpoint)
# ============================================================
GENERIC_PATH = "/kaggle/working/ifrnet_correction_generic_checkpoint.pth"

ckpt = torch.load(GENERIC_PATH, map_location="cpu")

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    state = ckpt["model_state_dict"]
    print("Full checkpoint. Trained through epoch:", ckpt.get("epoch", -1) + 1)
    for rec in ckpt.get("history", []):
        print(rec)
else:
    state = ckpt
    print("Raw weights (no epoch info).")

state = {k[len("module."):] if k.startswith("module.") else k: v
         for k, v in state.items()}

gen_net = CorrectionCNN(base_channels=32)
gen_net.load_state_dict(state, strict=True)
gen_net = gen_net.to(GPU0).eval()

print("\nGeneric correction model loaded.")

Full checkpoint. Trained through epoch: 4
{'epoch': 1, 'loss': 0.008410806043780555, 'psnr_base': 37.01021194458008, 'psnr_corrected': 37.00739669799805}
{'epoch': 2, 'loss': 0.008396438604753554, 'psnr_base': 37.01028823852539, 'psnr_corrected': 37.01414108276367}
{'epoch': 3, 'loss': 0.008392895941875892, 'psnr_base': 37.009742736816406, 'psnr_corrected': 37.0156364440918}
{'epoch': 4, 'loss': 0.008376853675628697, 'psnr_base': 37.01060485839844, 'psnr_corrected': 37.02893829345703}

Generic correction model loaded.


In [43]:
# ============================================================
# EVALUATE GENERIC CNN ON THE FINAL TEST SET
# ============================================================
import pandas as pd
import lpips
from skimage.metrics import structural_similarity as sk_ssim

SPLITS_PATH = "/kaggle/input/datasets/hibro00/vimeo-triplet-split/splits.csv"
GEN_OUT_CSV = "/kaggle/working/ifrnet_generic_results.csv"
MAX_CLIPS = None      # e.g. 100 for a quick trial run

_splits = pd.read_csv(SPLITS_PATH)
_splits["clip_id"] = _splits["clip_id"].astype(str)
gen_test_ids = list(dict.fromkeys(
    _splits.loc[_splits["split"].astype(str).str.lower() == "test", "clip_id"]
))
if MAX_CLIPS:
    gen_test_ids = gen_test_ids[:MAX_CLIPS]

gen_lpips = lpips.LPIPS(net="alex").to(GPU0).eval()

def _psnr(a, b):
    mse = torch.mean((a - b) ** 2).clamp_min(1e-10)
    return float((-10.0 * torch.log10(mse)).item())

def _ssim(a, b):
    a = a.squeeze(0).cpu().numpy().transpose(1, 2, 0)
    b = b.squeeze(0).cpu().numpy().transpose(1, 2, 0)
    return float(sk_ssim(b, a, channel_axis=2, data_range=1.0))

gen_results = []
print("Test clips:", len(gen_test_ids))

with torch.no_grad():
    for i, cid in enumerate(gen_test_ids):
        im0, gt, im1 = load_triplet(cid)
        im0, gt, im1 = im0.to(GPU0), gt.to(GPU0), im1.to(GPU0)

        ib = interp.run(im0, im1)                      # frozen IFRNet
        corrected, _, _ = gen_net(im0, im1, ib)
        corrected = corrected.clamp(0, 1)

        gen_results.append({
            "clip_id": cid,
            "gen_psnr": _psnr(corrected, gt),
            "gen_ssim": _ssim(corrected, gt),
            "gen_lpips": float(gen_lpips(corrected * 2 - 1, gt * 2 - 1).mean().item()),
        })

        if i == 0 or (i + 1) % 100 == 0:
            r = gen_results[-1]
            print(f"{i+1}/{len(gen_test_ids)} | {cid} | "
                  f"PSNR {r['gen_psnr']:.3f} | SSIM {r['gen_ssim']:.5f} | LPIPS {r['gen_lpips']:.5f}")
        if (i + 1) % 500 == 0:                          # crash-safe partial save
            pd.DataFrame(gen_results).to_csv(GEN_OUT_CSV, index=False)

gen_eval_df = pd.DataFrame(gen_results)
gen_eval_df.to_csv(GEN_OUT_CSV, index=False)
print("\nSaved:", GEN_OUT_CSV, "| rows:", len(gen_eval_df))

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Test clips: 3601
1/3601 | 00001/0389 | PSNR 38.861 | SSIM 0.98022 | LPIPS 0.01565
100/3601 | 00003/0436 | PSNR 36.027 | SSIM 0.97342 | LPIPS 0.02488
200/3601 | 00005/0258 | PSNR 37.603 | SSIM 0.98670 | LPIPS 0.00663
300/3601 | 00007/0017 | PSNR 34.467 | SSIM 0.94940 | LPIPS 0.03131
400/3601 | 00008/0179 | PSNR 38.900 | SSIM 0.96447 | LPIPS 0.02702
500/3601 | 00010/0207 | PSNR 40.242 | SSIM 0.98512 | LPIPS 0.01082
600/3601 | 00012/0457 | PSNR 37.205 | SSIM 0.97306 | LPIPS 0.01090
700/3601 | 00013/0988 | PSNR 29.090 | SSIM 0.94232 | LPIPS 0.01392
800/3601 | 00016/0554 | PSNR 36.997 | SSIM 0.96618 | LPIPS 0.04943
900/3601 | 00018/0731 | PSNR 29.556 | SSIM 0.96723 | LPIPS 0.02590
1000/3601 | 00020/0199 | PSNR 36.645 | SSIM 0.98396 | LPIPS 0.00840
1100/3601 | 00022/0601 | PSNR 32.012 | SSIM 0.96122 | LPIPS 0.01844
1200/3601 | 00024/0889 | PSNR 28.521 | SSIM 0.94788 | LPIPS 0.02694
1300/3601 | 00026/0848 